
# Aula 3 — KEMs Pós-Quânticos: ML-KEM (Kyber) e CB-KEM (HQC)

Este notebook acompanha a Aula 3 do curso *Fundamentos de Criptografia Pós-Quântica*. Ele foi preparado para estudo individual, sem a presença do professor, e também serve como roteiro de demonstrações para a aula síncrona.

Os códigos em SageMath neste notebook são modelos didáticos e não implementam ML-KEM e CB-KEM reais, mas preservam suas ideias estruturais: KeyGen / Encaps / Decaps, tolerância a ruído, derivação de chave e noções de “amostra típica”. Nos implementações reais, a derivação da chave e a robustez contra ataques ativos envolvem uma cadeia de funções (hash/PRF/KDF, verificações e validações), etapas de uma engenharia criptográfica responsáveis por garantir que:
- a chave derivada seja pseudorrandômica e uniforme;
- pequenas variações internas não gerem vazamentos;
- a decapsulação seja segura mesmo se o atacante manipular o texto cifrado (cápsula).

Neste notebook, deve permanecer claro que:
1. Encaps produz uma cápsula que, do ponto de vista do atacante, se comporta como uma amostra típica do problema (M)LWE;
2. Decaps usa o segredo para “remover” a parte correlacionada e recuperar um valor do qual se deriva a chave;
3. Hash/KDF (mesmo quando simplificado) é o mecanismo correto para transformar o material recuperado em uma chave simétrica apropriada.

Ao termino do estudo deste notebook, o aluno deverá produzir e entregar um Relatório com suas observações sobre as discussões e respostas às questões propostas neste notebook.

## Objetivos da aula

Ao final desta aula e deste notebook, você deverá ser capaz de:

- Explicar por que KEMs são uma abstração central para troca de chaves em protocolos modernos;
- Descrever a estrutura do ML-KEM em termos de KeyGen, Encaps e Decaps (como nos slides);
- Explicar por que a encapsulação “parece” uma nova amostra de MLWE para o adversário;
- Entender o papel de *hashing* na derivação da chave compartilhada (mesmo em modelos didáticos);
- Comparar, em alto nível, ML-KEM (reticulados) e CB-KEM (códigos) quanto às hipóteses e *trade-offs*.

## Síntese conceitual da aula

### O problema da troca de chaves no cenário pós-quântico

Protocolos modernos de segurança (TLS, SSH, VPNs) exigem o estabelecimento de um segredo compartilhado sobre um canal público. Historicamente, essa tarefa foi resolvida por Diffie–Hellman e ECC, cuja segurança repousa em problemas algébricos quebráveis pelo algoritmo de Shor.

No cenário pós-quântico, a troca de chaves clássica torna-se totalmente insegura, implicando perda completa de confidencialidade. Surge, portanto, a necessidade de substituir Diffie–Hellman por mecanismos resistentes a adversários quânticos.

### KEM como abstração criptográfica moderna

Os *Key Encapsulation Mechanisms* (KEMs) fornecem uma abstração moderna e robusta para troca de chaves:
- KeyGen gera um par de chaves pública/privada;
- Encaps(pk) produz um texto cifrado público (cápsula) e uma chave compartilhada;
- Decaps(sk, ct) recupera a mesma chave compartilhada.

Essa interface desacopla a troca de chaves da criptografia simétrica, permitindo que protocolos usem KEMs como substitutos diretos de Diffie–Hellman, sem alterar as camadas superiores. Por isso, KEMs são hoje a peça central da transição pós-quântica em protocolos reais.

### Segurança de KEMs: mais que dificuldade matemática

A segurança esperada de KEMs modernos é IND-CCA2, isto é, indistinguibilidade mesmo diante de ataques adaptativos com acesso a oráculos de decapsulação. Isso impõe requisitos que vão além da dificuldade matemática subjacente:
- o atacante pode modificar cápsulas;
- falhas de decapsulação não podem vazar informação;
- valores internos não devem ser usados diretamente como chaves.

Por essa razão, KEMs reais combinam matemática difícil com engenharia criptográfica cuidadosa, empregando funções de hash, KDFs e validações internas. Segurança de KEM não é apenas resolver um problema difícil, mas resistir a interação maliciosa.

### Onde entra o LWE / MLWE

A base matemática dominante dos KEMs pós-quânticos é o *Module-LWE* (MLWE). A ideia central é que:
- Alice e Bob obtêm valores correlacionados, mas não idênticos;
- o atacante observa apenas instâncias públicas que se assemelham a amostras típicas de MLWE;
- mecanismos de reconciliação implícita permitem que apenas as partes legítimas derivem a mesma chave.

O ruído cumpre papel duplo: protege contra o atacante e, ao mesmo tempo, exige mecanismos que garantam correção para Alice e Bob.

### ML-KEM (Kyber): MLWE transformado em protocolo real

O ML-KEM (Kyber) é o padrão do NIST para troca de chaves pós-quântica. Ele opera sobre vetores de polinômios em um anel quociente
$$
R_q = \mathbb{Z}_q[x]/(x^n + 1),
$$
com eficiência viabilizada pela NTT (Number Theoretic Transform).

Sua estrutura reflete diretamente o MLWE:
- a chave pública define uma instância pública do problema MLWE;
- o segredo é um vetor curto escondido por ruído;
- a encapsulação equivale à geração de uma nova amostra ruidosa;
- a decapsulação remove a correlação usando o segredo, permitindo recuperar uma mensagem e derivar a chave via hash.

Mecanismos internos garantem robustez contra ataques ativos (CCA): em caso de inconsistência, a chave derivada é aleatória, impedindo exploração adaptativa.

### CB-KEM (HQC): diversidade algorítmica baseada em códigos

O HQC representa uma abordagem alternativa baseada em códigos corretores de erros, explorando o problema de decodificação com erro. Embora a matemática seja distinta, a lógica conceitual é a mesma:
- o segredo é escondido em uma palavra de código;
- o atacante observa uma palavra ruidosa que parece aleatória;
- sem a estrutura secreta do código, a decodificação é intratável.

O HQC oferece diversidade algorítmica, mitigando riscos de dependência excessiva de uma única família matemática (reticulados).

### Comparação e implicações práticas

A comparação entre ML-KEM e HQC revela *trade-offs* claros:
- ML-KEM: chaves e cifrotextos menores, melhor desempenho, integração facilitada;
- CB-KEM: estruturas maiores e mais lentas, mas base matemática diferente.

Na prática, a transição ocorre por TLS híbrido, combinando esquemas clássicos e pós-quânticos. Para o usuário final, a PQC tende a ser onipresente e invisível, mas estruturalmente crítica.

### Mensagem central da aula

KEMs são o elo entre problemas matemáticos difíceis (como MLWE) e protocolos reais de troca de chaves.
A segurança pós-quântica nasce da matemática, mas só se concretiza por meio de engenharia criptográfica cuidadosa.


## Conexão com os slides

Leia os slides da Aula 3 e observe os seguintes conceitos:

- **KEM como abstração** → desacoplamento “troca de chave” ↔ “cifra simétrica” em TLS e afins;
- **Estrutura algorítmica do ML-KEM** → KeyGen / Encaps / Decaps descritos matematicamente;
- **Reconciliação / tolerância a ruído** → por que a correção depende de ruído controlado;
- **Derivação por hash** → por que a chave simétrica é derivada e não “usada crua”;
- **CB-KEM** → motivação de diversidade algorítmica; ideia de erro + decodificação em códigos.

Na seções que seguem, este notebook emprega células com código SageMath para auxiliar na compreensão prática de alguns dos conceitos abordados na aula, conforme o mapeamento a seguir:

- Slide "O que é um KEM?": Seção 1 (O que é um KEM?).
- Slide "Estrutura do ML-KEM: Decaps e Robustez": Seções 2 (Intuição para ML-KEM (Kyber)) e 2.1 (Taxa de sucesso).
- Slide "CB-KEM (HQC)": Seções 3 (Intuição para CB-KEM (HQC)), 3.1 (CB-KEM simplificado) e 3.2 (Efeito do número de erros).

## Instruções a seguir

**Antes de executar um célula de código**
- Identifique quais parâmetros você pode alterar, se houver.
- Antecipe qualitativamente o que deve acontecer.

**O que observar**
- Quais grandezas e saídas aparecem?
- Há algo que muda ao repetir a execução (aleatoriedade/semente)?

**O que registrar no Relatório**
- Resultados observados no experimento e sua interpretação sobre os mesmos.

## Preparação

Execute a célula de código a seguir para carregar bibliotecas e funções auxiliares. Depois, prossiga para a Seção 1.

In [4]:
# Notebook para kernel SageMath.
from sage.all import *
import random

ModuleNotFoundError: No module named 'sage'

## 1) O que é um KEM?

Um KEM tipicamente fornece três algoritmos:

- **KeyGen()** → (pk, sk)  
- **Encaps(pk)** → (ct, K)  
- **Decaps(sk, ct)** → K  

nos quais:
- `pk` = chave pública, `sk` = chave secreta,
- `ct` = *ciphertext* (cápsula),
- `K` = chave de sessão (simétrica) derivada.

Nesta seção, usaremos um KEM didático (não seguro) apenas para observar o fluxo.

In [ ]:
def toy0_keygen():
    sk = ZZ.random_element(1, 10)         # segredo pequeno (apenas didático)
    pk = 7*sk + 3                         # relação pública (didática)
    return pk, sk

def toy0_encaps(pk):
    K = ZZ.random_element(0, 1000)
    ct = K + pk
    return ct, K

def toy0_decaps(sk, ct):
    pk = 7*sk + 3
    K = ct - pk
    return K

pk, sk = toy0_keygen()
ct, K = toy0_encaps(pk)
K2 = toy0_decaps(sk, ct)

print("pk =", pk, "| sk =", sk)
print("ct =", ct)
print("K  =", K, "| K' =", K2, "| OK?", K == K2)

NameError: name 'ZZ' is not defined

### Discussão

Em que sentido um KEM abstrai a ideia de “troca de chaves” usada em protocolos criptográficos?

## 2) Intuição para ML-KEM (Kyber)

Nesta seção, será empregada aritmética modular (mod q) com ruído pequeno. Isto é apenas uma simplificação para entender o papel do ruído e a ideia de reconciliação.

In [ ]:
# Parâmetros didáticos
q = 97
n = 4
F = GF(q)

def small_noise(n, bound=1):
    return vector(ZZ, [ZZ.random_element(-bound, bound+1) for _ in range(n)])

def kem_lwe_keygen(q=97, n=4, bound=1):
    F = GF(q)
    A = random_matrix(F, n, n)
    s = random_vector(F, n)
    e = vector(F, [F(int(x)) for x in small_noise(n, bound)])
    t = A*s + e
    pk = (A, t)
    sk = s
    return pk, sk

def kem_lwe_encaps(pk, bound=1):
    (A, t) = pk
    q = A.base_ring().order()
    F = A.base_ring()
    n = A.nrows()

    r = random_vector(F, n)
    e1 = vector(F, [F(int(x)) for x in small_noise(n, bound)])
    e2 = F(int(ZZ.random_element(-bound, bound+1)))

    u = A.transpose()*r + e1
    v = (t.dot_product(r) + e2)

    v_int = ZZ(int(v)) % q
    if v_int > q//2: v_int -= q
    Kbit = 1 if v_int > 0 else 0

    ct = (u, v)
    return ct, Kbit, v_int

def kem_lwe_decaps(sk, pk, ct):
    s = sk
    (u, v) = ct
    q = pk[0].base_ring().order()

    vprime = v - (s.dot_product(u))
    vprime_int = ZZ(int(vprime)) % q
    if vprime_int > q//2: vprime_int -= q

    Kbit = 1 if vprime_int > 0 else 0
    return Kbit, vprime_int

pk, sk = kem_lwe_keygen(q=q, n=n, bound=1)
ct, K, v_int = kem_lwe_encaps(pk, bound=1)
K2, vprime_int = kem_lwe_decaps(sk, pk, ct)

print("K (encaps)   =", K, "| v_int =", v_int)
print("K (decaps)   =", K2, "| v'    =", vprime_int)
print("OK?", K == K2)

### 2.1 Taxa de sucesso

Nesta parte, medimos a taxa de sucesso do KEM didático: a proporção de execuções em que Alice e Bob chegam ao mesmo bit.

Variar `bound` pode não produzir uma mudança “limpa” e monotônica na taxa de sucesso.  Isso acontece porque o mecanismo foi propositalmente simplificado: a extração do bit usa uma decisão por limiar (*threshold*) e não há um mecanismo completo de reconciliação com dados auxiliares (*helper data*) como nos esquemas reais do ML‑KEM.  Assim, além do ruído explícito controlado por `bound`, outros efeitos podem dominar a discordância, como aritmética modular e valores próximos do limiar.

Para obter um resultado mais estável e interpretável:
- Aumente o número de testes (`trials`) para reduzir flutuações estatísticas (ex.: $500 \rightarrow 5000$).
- Fixe a semente do gerador aleatório antes do experimento se desejar reprodutibilidade;
- Registre a taxa de sucesso para vários valores de `bound` e discuta tendências, não apenas um número isolado.

Uma leitura correta do experimento seria:
- Se a taxa de sucesso cai claramente com `bound`, isso ilustra o *trade‑off* correção $\times$ ruído.  
- Se a taxa de sucesso parece pouco sensível a `bound`, isso ilustra por que KEMs reais precisam de reconciliação mais cuidadosa: uma simplificação sem “zonas de tolerância” pode ter falhas dominadas por limiares e efeitos modulares.

**Observação importante sobre o uso de hash**

Em esquemas pós-quânticos padronizados, como o ML-KEM (Kyber), o valor obtido ao final da encapsulação nunca é usado diretamente como chave criptográfica. Em vez disso, esse valor intermediário é passado por uma função de hash ou KDF, que produz uma chave final com distribuição uniforme e garante segurança contra ataques adaptativos.

Neste notebook, optamos por não usar hash de forma deliberada. Essa escolha permite observar diretamente:
- como o ruído afeta a concordância de chave;
- quando e por que a correção falha;
- por que mecanismos adicionais são necessários em esquemas reais.

O objetivo aqui não é reproduzir a estrutura o ML-KEM, mas expor o núcleo do problema de concordância sob ruído, sem camadas adicionais de proteção.

In [ ]:
def lwe_toy_success(trials=200, bound=1):
    ok = 0
    for _ in range(trials):
        pk, sk = kem_lwe_keygen(q=q, n=n, bound=bound)
        ct, K, _ = kem_lwe_encaps(pk, bound=bound)
        K2, _ = kem_lwe_decaps(sk, pk, ct)
        if K == K2:
            ok += 1
    return ok, trials, ok/trials

for b in [0, 1, 2]:
    ok, tr, rate = lwe_toy_success(trials=300, bound=b)
    print(f"bound={b}: sucesso={ok}/{tr} = {rate:.3f}")

NameError: name 'kem_lwe_keygen' is not defined

### Discussão

O que significa “taxa de sucesso” neste experimento? Quando dizemos que houve “sucesso” ou “falha”?

Por que `bound` pode não controlar a taxa de sucesso de forma clara neste experimento? A partir do que você observou, descreva pelo menos dois fatores, além do ruído explícito controlado por `bound`, que podem causar discordância entre Alice e Bob. Pense em decisões por limiar (*threshold*), aritmética modular (*wrap‑around*) e ausência de reconciliação completa com dados auxiliares.

Em esquemas reais como o ML-KEM, a chave compartilhada é derivada por meio de funções de hash ou KDF. Considerando os experimentos realizados neste notebook, que informações sobre correção e ruído seriam ocultadas se uma função de hash fosse aplicada imediatamente?

## 3) Intuição para CB-KEM (HQC)

Usaremos um código de Hamming (7,4) para intuição: 4 bits são codificados em 7, erros são adicionados, seguindo a decodificação.

In [ ]:
# Código de Hamming (7,4) em GF(2)
F2 = GF(2)

G = matrix(F2, [
    [1,0,0,0, 0,1,1],
    [0,1,0,0, 1,0,1],
    [0,0,1,0, 1,1,0],
    [0,0,0,1, 1,1,1],
])

H = matrix(F2, [
    [0,1,1,1, 1,0,0],
    [1,0,1,1, 0,1,0],
    [1,1,0,1, 0,0,1],
])

def hamming74_encode(m4):
    return m4 * G

# tabela de síndromes (correção de 1 erro)
syn_to_pos = {}
for pos in range(7):
    e = vector(F2, [1 if i==pos else 0 for i in range(7)])
    syn = tuple((H*e.column()).list())
    syn_to_pos[syn] = pos

def hamming74_decode(r7):
    syn = tuple((H * r7.column()).list())
    if syn == (0,0,0):
        c = r7
        corrected = False
    else:
        if syn in syn_to_pos:
            pos = syn_to_pos[syn]
            c = r7 + vector(F2, [1 if i==pos else 0 for i in range(7)])
            corrected = True
        else:
            c = r7
            corrected = False
    m = vector(F2, c[:4])
    return m, c, corrected, syn

# teste rápido
m = vector(F2, [1,0,1,1])
c = hamming74_encode(m)
pos = randint(0,6)
r = c + vector(F2, [1 if i==pos else 0 for i in range(7)])
m_hat, c_hat, corrected, syn = hamming74_decode(r)

print("m =", m)
print("c =", c)
print("erro em pos", pos, "-> r =", r)
print("m_hat =", m_hat, "| corrigiu?", corrected, "| OK?", m_hat == m)


NameError: name 'GF' is not defined

### 3.1 CB-KEM simplificado

Encapsula uma chave `m` (4 bits) por codificação e adição de erro. Em seguida, decapsula por decodificação.

In [ ]:
def kem_code_keygen():
    pk = "Hamming(7,4)-public"
    sk = "Hamming(7,4)-secret"
    return pk, sk

def kem_code_encaps(pk, error_weight=1):
    m = vector(F2, [randint(0,1) for _ in range(4)])
    c = hamming74_encode(m)
    r = c
    positions = sample(range(7), error_weight)
    for pos in positions:
        r = r + vector(F2, [1 if i==pos else 0 for i in range(7)])
    ct = r
    K = m
    return ct, K, positions

def kem_code_decaps(sk, ct):
    m_hat, c_hat, corrected, syn = hamming74_decode(ct)
    return m_hat, corrected, syn

pk, sk = kem_code_keygen()
ct, K, pos = kem_code_encaps(pk, error_weight=1)
K2, corrected, syn = kem_code_decaps(sk, ct)

print("erros em pos", pos)
print("K  =", K)
print("K' =", K2, "| corrigiu?", corrected, "| OK?", K2 == K)


### 3.2 Efeito do número de erros

O código de Hamming (7,4) corrige 1 erro. A célula a seguir visa medir a taxa de sucesso para 0, 1 ou 2 erros.

In [ ]:
def code_toy_success(trials=300, error_weight=1):
    ok = 0
    corr = 0
    pk, sk = kem_code_keygen()
    for _ in range(trials):
        ct, K, _ = kem_code_encaps(pk, error_weight=error_weight)
        K2, corrected, _ = kem_code_decaps(sk, ct)
        if corrected:
            corr += 1
        if K2 == K:
            ok += 1
    return ok/trials, corr/trials

for w in [0, 1, 2, 3]:
    ok_rate, corr_rate = code_toy_success(trials=500, error_weight=w)
    print(f"erros={w}: sucesso={ok_rate:.3f} | 'corrigiu'={corr_rate:.3f}")


NameError: name 'kem_code_keygen' is not defined

### Discussão

Qual é o papel central dos códigos corretores de erros na intuição por trás do HQC?  

O que acontece quando o número de erros excede a capacidade de correção?

## 5) Exploração adicional

- **LWE simplificado:** varie `bound`, mas faça isso com muitos testes (`trials`) e, se quiser, com semente fixa para comparar cenários. Observe se há tendência e discuta o resultado à luz da nota da Seção 2.1: em uma implementação simplificada, sem reconciliação completa, o efeito de `bound` pode ficar mascarado por decisões por limiar e efeitos modulares.
- **CB-KEM simplificado:** varie `error_weight` e observe a “quebra” quando o número de erros ultrapassa a capacidade de correção (ex.: Hamming(7,4) corrige 1 erro).  
  Aqui, em geral, a transição é mais visível, pois a relação “erros → falha” é mais direta.

### Discussão

Como esquemas reais escolhem parâmetros para obter falha desprezível e desempenho aceitável?


## Questões finais para o Relatório

Responda às questões abaixo com base nos slides e neste notebook.

1. Explique o papel de cada uma das funções KeyGen, Encaps e Decaps no ML-KEM.
2. Por que encapsular é equivalente a gerar uma nova amostra do problema MLWE?
3. Qual é o papel da função hash/KDF na derivação da chave compartilhada? O que se espera garantir com isso?
4. Compare, em alto nível, ML-KEM e CB-KEM quanto às hipóteses matemáticas de segurança e *trade-offs* (tamanho, desempenho, implementação).
5. Por que KEMs permitem desacoplar* troca de chaves e criptografia simétrica em protocolos?
6. Do ponto de vista prático: se a taxa de sucesso de decapsulação cai ao aumentar ruído/limites, o que isso indica para uso em protocolos reais? Se essa taxa não cair no modelo simplificado aqui apresentado, como você explicaria essa limitação?